In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!ls /kaggle/input/grammar-audios

ls: cannot access '/kaggle/input/grammar-audios': No such file or directory


In [3]:
from IPython.display import display
from ipywidgets import FileUpload

# Create upload widget
upload = FileUpload(accept='.wav', multiple=False)
display(upload)

FileUpload(value=(), accept='.wav', description='Upload')

In [4]:
# Get the uploaded file content
uploaded_file = list(upload.value.values())[0]
content = uploaded_file['content']

# Save it as Sample.wav in the working folder
with open('/kaggle/working/Sample.wav', 'wb') as f:
    f.write(content)

audio_path = '/kaggle/working/Sample.wav'
print("Audio saved to:", audio_path)

AttributeError: 'tuple' object has no attribute 'values'

In [5]:
from IPython.display import display
from ipywidgets import FileUpload

upload = FileUpload(accept='.wav', multiple=False)
display(upload)

FileUpload(value=(), accept='.wav', description='Upload')

In [6]:
import io

# For Kaggle, the upload returns a tuple
if isinstance(upload.value, tuple):
    uploaded_file = upload.value[0]  # first uploaded file
    content = uploaded_file['content']
else:
    uploaded_file = list(upload.value.values())[0]
    content = uploaded_file['content']

# Save to working folder
with open('/kaggle/working/Sample.wav', 'wb') as f:
    f.write(content)

audio_path = '/kaggle/working/Sample.wav'
print("Audio saved to:", audio_path)

Audio saved to: /kaggle/working/Sample.wav


In [7]:
!pip install --quiet gradio openai-whisper language-tool-python
!apt-get install -qq ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.6 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.1 MB/s eta 0:00:00:00:0100:01


In [8]:
import gradio as gr
import whisper
from language_tool_python import LanguageTool

# Load Whisper model (this may take a minute)
model = whisper.load_model("base")
tool = LanguageTool('en-US')

# Function to handle uploaded audio
def transcribe_grammar(audio):
    # audio is a temp file from Gradio
    result = model.transcribe(audio.name)
    text = result["text"]
    
    matches = tool.check(text)
    grammar_score = max(0, 10 - len(matches))  # simple scoring
    
    # Return transcription, grammar score, first 5 issues
    sample_issues = [(m.ruleId, m.message) for m in matches[:5]]
    return text, grammar_score, sample_issues

# Create Gradio interface
iface = gr.Interface(
    fn=transcribe_grammar,
    inputs=gr.Audio(type="file"),
    outputs=["text", "number", "json"],
    title="Audio Grammar Checker",
    description="Upload a .wav file, and this app will transcribe it and give a grammar score."
)

# Launch the app with a public shareable link
iface.launch(share=True)

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 154MiB/s]


ValueError: Invalid value for parameter `type`: file. Please choose from one of: numpy filepath

In [10]:
gr.Audio(type="filepath")

In [11]:
import gradio as gr
import whisper
from language_tool_python import LanguageTool

# Load Whisper model
model = whisper.load_model("base")
tool = LanguageTool('en-US')

def transcribe_grammar(audio_path):
    # audio_path is now a string path
    result = model.transcribe(audio_path)
    text = result["text"]
    
    matches = tool.check(text)
    grammar_score = max(0, 10 - len(matches))
    
    # Return transcription, grammar score, first 5 issues
    sample_issues = [(m.ruleId, m.message) for m in matches[:5]]
    return text, grammar_score, sample_issues

iface = gr.Interface(
    fn=transcribe_grammar,
    inputs=gr.Audio(type="filepath"),  # FIXED here
    outputs=["text", "number", "json"],
    title="Audio Grammar Checker",
    description="Upload a .wav file, and this app will transcribe it and give a grammar score."
)

# Launch app with public link
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://34c417b96120f25f25.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [13]:
!pip install --quiet gradio openai-whisper language-tool-python
!apt-get install -qq ffmpeg

In [14]:
import gradio as gr
import whisper
from language_tool_python import LanguageTool

# Load models
model = whisper.load_model("base")
tool = LanguageTool('en-US')

def transcribe_grammar(audio_path):
    result = model.transcribe(audio_path)
    text = result["text"]
    matches = tool.check(text)
    grammar_score = max(0, 10 - len(matches))
    sample_issues = [(m.ruleId, m.message) for m in matches[:5]]
    return text, grammar_score, sample_issues

iface = gr.Interface(
    fn=transcribe_grammar,
    inputs=gr.Audio(type="filepath"),
    outputs=["text", "number", "json"],
    title="Audio Grammar Checker",
    description="Upload a .wav file, and this app will transcribe it and give a grammar score."
)

iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://08d472aec602c4c455.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
